# Model Module Sanity Notebook

This notebook validates model-related components:

1. Model build + forward shapes
2. Optional feature outputs
3. LoRA injection and layer-wise rank schedule
4. Trainable parameter modes (`full_finetune` vs `lora_finetune`)
5. Real batch forward from data loader


## Notes

- We force `pretrained=False` in sanity tests to avoid external weight download.
- These tests are for correctness and interface stability, not benchmark accuracy.

In [4]:
import sys
sys.path.append('..')

from copy import deepcopy
from pathlib import Path
import torch

from src.config import load_config
from src.models import build_model, get_param_groups
from src.data.utils import build_pretrain_loaders
from src.models.lora import apply_finetune_mode

CFG_SRC = Path('../configs/train_source_office_home.yaml')
CFG_FT = Path('../configs/finetune_target_office_home.yaml')
cfg_src = load_config(CFG_SRC)
cfg_ft = load_config(CFG_FT)
print('Loaded configs:', CFG_SRC, CFG_FT)


Loaded configs: ../configs/train_source_office_home.yaml ../configs/finetune_target_office_home.yaml


## 1) Build Model + Forward (No LoRA)

In [2]:
cfg = deepcopy(cfg_src)
cfg.model.pretrained = False
model = build_model(cfg)
model.eval()

x = torch.randn(4, 3, int(cfg.data.image_size), int(cfg.data.image_size))
with torch.no_grad():
    logits = model(x)
print('logits shape:', tuple(logits.shape))
assert tuple(logits.shape) == (4, int(cfg.data.num_classes))
print('Basic forward check passed.')


logits shape: (4, 65)
Basic forward check passed.


In [3]:
model

ResNetBottleneckClassifier(
  (stem): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  )
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(6

## 2) Forward with Features

In [4]:
with torch.no_grad():
    out = model(x, return_features=True)
print('keys:', sorted(out.keys()))
assert set(out.keys()) == {'logits', 'feat_2048', 'feat_256'}
assert tuple(out['logits'].shape) == (4, int(cfg.data.num_classes))
assert tuple(out['feat_2048'].shape) == (4, 2048)
assert out['feat_256'].shape[0] == 4
print('Feature forward check passed.')


keys: ['feat_2048', 'feat_256', 'logits']
Feature forward check passed.


## 3) LoRA Injection + Layer-wise Rank Schedule

In [5]:
cfg_lora = deepcopy(cfg_ft)
cfg_lora.model.pretrained = False
model_lora = build_model(cfg_lora)

lora_param_names = [n for n, p in model_lora.named_parameters() if 'lora_' in n]
print('num lora params:', len(lora_param_names))
assert len(lora_param_names) > 0, 'LoRA params not found; injection failed'

# Check rank schedule stored in PEFT config
peft_cfg = model_lora.peft_config['default']
rank_pattern = getattr(peft_cfg, 'rank_pattern', None)
print('rank_pattern type:', type(rank_pattern).__name__)
assert isinstance(rank_pattern, dict) and len(rank_pattern) > 0

ranks = set(rank_pattern.values())
print('distinct assigned ranks:', sorted(ranks))
assert ranks.issuperset({16, 8, 4, 2})
print('LoRA rank schedule check passed.')


num lora params: 92
rank_pattern type: dict
distinct assigned ranks: [2, 4, 8, 16]
LoRA rank schedule check passed.


In [6]:
model_lora

PeftModel(
  (base_model): LoraModel(
    (model): ResNetBottleneckClassifier(
      (stem): Sequential(
        (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      )
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): lora.Conv2d(
            (base_layer): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (lora_dropout): ModuleDict(
              (default): Identity()
            )
            (lora_A): ModuleDict(
              (default): Conv2d(64, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
            )
            (lora_B): ModuleDict(
              (default): Conv2d(16, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
            )
            (lora_embedding_A): ParameterDict()
   

## 4) Trainable Parameter Modes

In [7]:
# full_finetune
model_full = build_model(cfg)
apply_finetune_mode(model_full, mode='full_finetune')
n_full = sum(p.numel() for p in model_full.parameters() if p.requires_grad)
n_all = sum(p.numel() for p in model_full.parameters())
print('full_finetune trainable/total params:', n_full, '/', n_all)
print('full_finetune ratio:', f'{n_full / n_all:.4f}')
assert n_full == n_all, 'full_finetune should unfreeze all parameters'

# lora_finetune
model_lf = build_model(cfg_lora)
apply_finetune_mode(model_lf, mode='lora_finetune')
trainable = [n for n, p in model_lf.named_parameters() if p.requires_grad]
print('lora_finetune trainable count:', len(trainable))
assert any('lora_' in n for n in trainable)
n_lora = sum(p.numel() for p in model_lf.parameters() if p.requires_grad)
n_all_lora = sum(p.numel() for p in model_lf.parameters())
print('lora_finetune trainable/total params:', n_lora, '/', n_all_lora)
print('lora_finetune ratio:', f'{n_lora / n_all_lora:.4f}')
assert n_lora < n_all_lora, 'lora_finetune should train only a subset of parameters'
print('Trainable mode checks passed.')


full_finetune trainable/total params: 24049793 / 24049793
full_finetune ratio: 1.0000
lora_finetune trainable count: 120
lora_finetune trainable/total params: 118016 / 24122497
lora_finetune ratio: 0.0049
Trainable mode checks passed.


## 5) Real Batch Forward from Data Loader

In [8]:
cfg_data = deepcopy(cfg_src)
cfg_data.train.num_workers = 0
cfg_data.model.pretrained = False
loaders = build_pretrain_loaders(cfg_data)
batch = next(iter(loaders['source_train']))
print('batch keys:', sorted(batch.keys()))

model_data = build_model(cfg_data)
model_data.eval()
with torch.no_grad():
    logits = model_data(batch['image'])
print('batch forward logits shape:', tuple(logits.shape))
assert logits.shape[0] == batch['image'].shape[0]
assert logits.shape[1] == int(cfg_data.data.num_classes)
print('Data+model forward check passed.')


batch keys: ['image', 'label', 'sample_id']
batch forward logits shape: (64, 65)
Data+model forward check passed.
